In [1]:
# Save file list - adjust based on climate model and location of files

In [4]:
import os
import json
from datetime import datetime
from utils.utils import get_scenario_config

In [8]:
# === Filename builders for CESM2 ===
def cesm_file_list(scenario, ens_num):
    num = f"{ens_num:02d}"
    files = []

    if scenario == "ARISE":
        end = "206912" if ens_num not in [8, 9] else "207012"
        path = os.path.join(
            "/glade/campaign/cesm/collections/ARISE-SAI-1.5/",
            f"b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.0{num}/atm/proc/tseries/month_1/",
            f"b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.0{num}.cam.h0.PM25.203501-{end}.nc"
        )
        files.append(path)

    elif scenario == "SSP245":
        base = os.path.join(
            "/glade/campaign/cesm/collections/CESM2-WACCM-SSP245/",
            f"b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.0{num}/atm/proc/tseries/month_1/"
        )
        files.append(os.path.join(base, f"b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.0{num}.cam.h0.PM25.201501-206412.nc"))
        end = "210012" if ens_num <= 5 else "206912"
        files.append(os.path.join(base, f"b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.0{num}.cam.h0.PM25.206501-{end}.nc"))

    elif scenario == "SSP245_G6":
        base = os.path.join(
            "/glade/campaign/cesm/collections/CESM2-WACCM-SSP245/",
            f"b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.0{num}/atm/proc/tseries/month_1/"
        )
        files.append(os.path.join(base, f"b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.0{num}.cam.h0.PM25.201501-206412.nc"))
        end = "210012" if ens_num <= 5 else "206912"
        files.append(os.path.join(base, f"b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.0{num}.cam.h0.PM25.206501-{end}.nc"))

    elif scenario == "G6-1.5K":
        path = os.path.join(
            "/glade/campaign/collections/rda/data/d651059/ARISE-SAI-1.5/",
            f"b.e21.BW.f09_g17.SSP245-G6-1p5K-SAI.0{num}/atm/proc/tseries/month_1/",
            f"b.e21.BW.f09_g17.SSP245-G6-1p5K-SAI.0{num}.cam.h0.PM25.203501-208412.nc"
        )
        files.append(path)

    elif scenario == "hist":
        base = os.path.join(
            "/glade/campaign/cesm/development/wawg/WACCM6-TSMLT-HIST/"
            f"b.e21.BWHISTcmip6.f09_g17.CMIP6-historical-WACCM.1980_2014.0{num}/atm/proc/tseries/month_1/"
        )
        files.append(os.path.join(base, f"b.e21.BWHISTcmip6.f09_g17.CMIP6-historical-WACCM.1980_2014.0{num}.cam.h0.PM25.197801-199912.nc"))
        files.append(os.path.join(base, f"b.e21.BWHISTcmip6.f09_g17.CMIP6-historical-WACCM.1980_2014.0{num}.cam.h0.PM25.199912-201412.nc"))

    return files

In [2]:
# === Filename builders for UKESM1 ===
# CHANGE WHEN UKESM1 DATA IS AVAILABLE FOR PM2.5
def ukesm_file_list(scneario, ens_num):
    files = []
    file_path = f"/glade/work/awells/air_quality/UKESM1/data/{scenario}/"
    if scenario == "hist":
        files.append(os.path.join(file_path, "sfo3_AERhr_UKESM1-0-LL_historical_r1i1p1f2_gn_199001010030-199912302330.nc"))
        files.append(os.path.join(file_path, "sfo3_AERhr_UKESM1-0-LL_historical_r1i1p1f2_gn_200001010030-200912302330.nc"))
        files.append(os.path.join(file_path, "sfo3_AERhr_UKESM1-0-LL_historical_r1i1p1f2_gn_201001010030-201412302330.nc"))

    else:
        # one file per scenario and ensemble
        files.append(os.path.join(file_path, f"ukesm_ssp_o3_3hr_{ens_num:02d}.nc"))

    return files

In [3]:
def get_file_list(model, scenario, ens_num):
    try:
        return MODEL_HANDLERS[model](scenario, ens_num)
    except KeyError:
        raise ValueError(
            f"Model {model} not supported. Options: {list(MODEL_HANDLERS)}"
        )

In [5]:
def save_file_list_with_metadata(model, scenario, ens_num, file_list, DIR, filename):
    data = {
        "model": model,
        "scenario": scenario,
        "ensemble_number": ens_num,
        "generated_on": datetime.now().isoformat(),
        "files": file_list
    }
    file_path = os.path.join(DIR, filename)
    with open(file_path, "w") as f:
        json.dump(data, f, indent=2)

In [11]:
# === Master lookup ===
MODEL_HANDLERS = {
    "CESM2": cesm_file_list,
    "UKESM1": ukesm_file_list,
}

# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "CESM2"
scenario = "G6-1.5K"

config = get_scenario_config(model, scenario)
ensemble_members = config["ensemble_members"]
years = config["years"]

SAVE_DIR = f"/glade/work/awells/air_quality/{model}/pm25/file_paths/"

for ens_num in ensemble_members:
    print(f"Building list of files for {scenario}, Ensemble {ens_num:02d}")
    file_list = get_file_list(model, scenario, ens_num)
    save_file_list_with_metadata(
        model,
        scenario,
        ens_num,
        file_list,
        SAVE_DIR,
        f"file_list_{scenario}_{ens_num}.json"
    )

print("All processing complete.")

Building list of files for G6-1.5K, Ensemble 01
Building list of files for G6-1.5K, Ensemble 02
Building list of files for G6-1.5K, Ensemble 03
All processing complete.
